# Crowd counting with MCNN — Colab training

Runs the whole pipeline on a Colab GPU: dataset download, preprocessing,
training, evaluation and the Gradio demo.

**Before you start:** `Runtime -> Change runtime type -> GPU`.

Checkpoints are written to Google Drive, so a disconnect costs at most
one epoch — rerun the resume cell and training picks up where it stopped.

## 1. Check the GPU

In [ ]:
!nvidia-smi

import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Mount Google Drive

Checkpoints and logs live here so they survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/crowd-counting-mcnn'
!mkdir -p "$DRIVE_DIR/checkpoints"
print('checkpoints ->', DRIVE_DIR + '/checkpoints')

## 3. Clone the repository and install dependencies

In [ ]:
REPO_URL = 'https://github.com/jugalkkt/crowd-density-maps.git'

!git clone -q $REPO_URL /content/crowd-counting-mcnn
!pip install -q -r /content/crowd-counting-mcnn/requirements.txt

In [ ]:
%cd /content/crowd-counting-mcnn
!ls

## 4. Download ShanghaiTech

Uses the Kaggle API. Get `kaggle.json` from your Kaggle account page
(*Settings -> API -> Create New Token*) and upload it when prompted.

In [ ]:
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json:')
    files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
!pip install -q kaggle
!mkdir -p /content/data/raw
!kaggle datasets download -d tthien/shanghaitech -p /content/data --unzip

In [ ]:
# The archive layout differs between mirrors; point raw_data_dir at
# whatever directory actually contains part_A / part_B.
!find /content/data -maxdepth 3 -iname 'part_*' -type d

## 5. Preprocess both parts

Generates density maps: adaptive sigma for Part A, fixed sigma for Part B.

In [ ]:
RAW_DIR = '/content/data'            # edit if the find above shows another path
PROCESSED_DIR = '/content/data/processed'
DRIVE_CKPT = DRIVE_DIR + '/checkpoints'

!python -m src.data.preprocess --part A --verify \
    --set paths.raw_data_dir=$RAW_DIR \
    --set paths.processed_dir=$PROCESSED_DIR \
    --set density.mode=adaptive --visualize 3

In [ ]:
!python -m src.data.preprocess --part B --verify \
    --set paths.raw_data_dir=$RAW_DIR \
    --set paths.processed_dir=$PROCESSED_DIR \
    --set density.mode=fixed --visualize 3

In [ ]:
# Sanity check: blobs should sit on people's heads.
from IPython.display import Image, display
import glob

for path in sorted(glob.glob('outputs/density_samples/*.png'))[:4]:
    display(Image(path))

## 6. Train

Part B first — it is smaller and converges faster, so it is the better
smoke test. Watch `val MAE` fall over the first ~20 epochs.

If the model collapses to predicting zeros, raise `train.density_scale`
(try 100) and/or `train.lr`.

In [ ]:
PART = 'B'

!python -m src.train \
    --set data.part=$PART \
    --set paths.processed_dir=$PROCESSED_DIR \
    --set paths.checkpoint_dir=$DRIVE_CKPT \
    --set train.epochs=400 \
    --set train.lr=1e-5 \
    --set train.num_workers=2

### Resume after a disconnect

Re-run the install and mount cells, then this one.

In [ ]:
!python -m src.train \
    --set data.part=$PART \
    --set paths.processed_dir=$PROCESSED_DIR \
    --set paths.checkpoint_dir=$DRIVE_CKPT \
    --resume $DRIVE_CKPT/part_$PART/last.pth

### Watch training in TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $DRIVE_CKPT

## 7. Evaluate on the official test split

In [ ]:
!python -m src.evaluate \
    --checkpoint $DRIVE_CKPT/part_$PART/best.pth \
    --part $PART --save-predictions \
    --set paths.processed_dir=$PROCESSED_DIR

In [ ]:
from IPython.display import Image, display
import json, glob

print(json.dumps(json.load(open(f'outputs/eval_part_{PART}.json')), indent=2))
display(Image(f'outputs/scatter_part_{PART}.png'))
for path in sorted(glob.glob(f'outputs/worst_part_{PART}/*.png'))[:3]:
    display(Image(path))

## 8. Launch the Gradio demo

`--share` gives a public URL that works from outside Colab.

In [ ]:
!python app/gradio_app.py --share \
    --checkpoint-b $DRIVE_CKPT/part_B/best.pth \
    --checkpoint-a $DRIVE_CKPT/part_A/best.pth